# 2. Create KItems with the SDK

In this tutorial we see how to create new KItems.

### 2.1. Setting up
Before you run this tutorial: make sure to have access to a DSMS-instance of your interest, along with installation of this package, and have established access to the DSMS through DSMS-SDK (refer to [Connecting to DSMS](../dsms_sdk.md#connecting-to-dsms))

Now let us import the needed classes and functions for this tutorial.

In [ ]:
from dsms import DSMS, KItem

Now source the environmental variables from an `.env` file and start the DSMS-session.

In [ ]:
import os
dsms = DSMS(env=".env") if os.path.exists(".env") else DSMS()


### 2.2. Create KItems

We can make new KItems by simple class-initiation. (Do not pass an existing KItem as input, as this will raise an error.)

In [ ]:
item = KItem(
    name="Specimen123",
    ktype_id=dsms.ktypes.Specimen,
    custom_properties = {
        "Width": 0.5,
        "Length": 0.15,
    }
)

item

Remember: changes are only synchronized with the DSMS when you call the `commit`-method:

In [ ]:
dsms.add(item)
dsms.commit()
item.url

As we can see, the object we created before running the `commit`-method has automatically been updated, e.g. with the creation- and update-timestamp. We can check this with the below command:

In [ ]:
item

To just get the name of the item, we can do it as follows:

In [ ]:
item.name

As well as the id of the KItem we can do it as follows:

In [ ]:
item.id

To check the KType of the item newly created we can use the following:

In [ ]:
item.ktype

... and also check the KType:

In [ ]:
item.is_a(dsms.ktypes.Specimen)

We are able to print the subgraph related to the KItem:

In [ ]:
try:
    print(item.subgraph.serialize())
except ValueError as e:
    print(f"Note: RDF subgraph is generated asynchronously.")
    print(f"It may not be available immediately after creation.")

And we can convert the units:

In [ ]:
try:
    item.custom_properties.Width.convert_to("m")
except ValueError as e:
    print(f"Unit conversion not available: {e}")

In [ ]:
try:
    item.custom_properties.Length.convert_to("m")
except ValueError as e:
    print(f"Unit conversion not available: {e}")

You can also convert the custom_properties to a flat dict by passing the `flat`-parameter to the `model_dump`-method of the pydantic model:

In [ ]:
item.custom_properties.model_dump(flat=True)

### 2.x. Setting access properties

Access control is defined via `access_properties`, which assigns roles to specific users and groups. The available roles are `MEMBER` (read only), `CONTRIBUTOR` (read and update), `OWNER` (read, update, delete, manage), and `ADMIN` (same as OWNER).

In [ ]:
from dsms.knowledge.properties.access import KItemAccessProperties, Role

# Look up the current user to demonstrate role assignment
uname = dsms.config.username
if hasattr(uname, "get_secret_value"):
    uname = uname.get_secret_value()
current_user = dsms.users.by_username.get(uname)

item.access_properties = KItemAccessProperties(
    user_access=[{"user_id": current_user.id, "role": Role.OWNER}],
)
dsms.commit()


Now you can check if the particular KItem is in the list of KItems. This can be done either by using the command:
    `
     dsms.kitems
    `
    or by logging into the frontend dsms instance.